# Niche model — vector overlays (03b)

Reads the generalised NiN layer from **`03a_nisjemodell_raster.ipynb`** and applies
the vector overlays:

tørrfall → observation substrate → clip to grunnlinje → subtract land →
**checkpoint (parquet)** → usikkerhet, NiN type and Naturkart attributes → save.

Each polygon carries the four axes plus `datakilde`; the overlays only change the
axes they are responsible for, and the NiN type is derived once at the end.

The checkpoint after land subtraction lets you rerun the attribute step (reload
cell) without rerunning the overlays.

In [1]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

import mnk
import mnk.attributes
import mnk.utils
import mnk.vectorize
import mnk.rules

CRS = "EPSG:25833"
EPSG = 25833
NODATA = 255

## Sources and paths

In [ ]:
# Local files live in the repository root (Jupyter runs from notebooks/, papermill from the root).
REPO = mnk.utils.repo_root()

# Inngang: generalisert NiN-lag fra 03a.
nin_vec_path = "gs://niva-geodata/MarintNaturKart/results/nisjedata-modell_norge_2026_25833_generalized.gpkg"

# Observasjons-substrat (autoritativt, vektor-overlay).
obs_substrat_parquet = (
    "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-klassifisering_norge_latest_25833.geo.parquet"
)
# Tørrfall med innelukket vann (prepped av mnk.sources.prepare_torrfall_filled).
torrfall_parquet = "gs://niva-geodata/MarintNaturKart/results/torrfall-filled_norge_25833.geo.parquet"
land_parquet = "gs://niva-geodata/MarintNaturKart/aux/land-mask-norway_FKB-osm-seaborder-subdivided-fullres_v3_25833_20260924.geo.parquet"

# Drivere for usikkerhet (per-akse beslutningsmargin, NKM_nisjedata_kvalitet):
# substratsannsynlighet, Kd490 (lys), dybde (lys), og salinitet.
subprob_path = "gs://niva-geodata/MarintNaturKart/results/3band_probability_calibrated.tif"
kd_path = "gs://niva-geodata/MarintNaturKart/results/KD490_norge_2020-2025_filled_25833.tif"
depth_path = "gs://niva-geodata/MarintNaturKart/features/feature_norge_dem50_depth_filled.tif"
salinity_path = "gs://niva-geodata/MarintNaturKart/input/hi/Norfjords_160_surf_salt_mean_UTM33.tif"

# Checkpoint (kun parquet) etter klipp/land, før attributter.
clipped_checkpoint = REPO / "nisjemodell_norge_2026_25833_clipped.geo.parquet"

## Axes

Each polygon carries the four axes as data-model codes — `nkm_lys`, `nkm_lm-sa`,
`nkm_lm-dk`, `nkm_lm-tv` (see `mnk.rules.NKM`) — and `datakilde`, one source token per
axis. The 03a raster class (1-12, `mnk.rules.NIN_INT_AXES`) is decoded once on load
with `mnk.attributes.axes_from_nin_int`. `nin_kode_fkid`/`NiN_name` are derived once at
the end with `mnk.attributes.classify`.

In [ ]:
LYS, SAL, SUB, TF = (
    mnk.rules.COL_LYS, mnk.rules.COL_SALINITET, mnk.rules.COL_SUBSTRAT, mnk.rules.COL_TOERFALL
)

# NGU BunnType -> substrate code (blanding counts as soft bottom).
BUNN_TO_SUBSTRAT = {
    "løsbunn": mnk.rules.NKM_SUBSTRAT["løst"],
    "blanding": mnk.rules.NKM_SUBSTRAT["løst"],
    "fastbunn": mnk.rules.NKM_SUBSTRAT["fast"],
}

## Load NiN layer (from 03a) + merge slivers

In [ ]:
gdf_nin = gpd.read_file(nin_vec_path)
gdf_nin = gdf_nin.rename(columns={"class_int": "nin_int"})[["nin_int", "geometry"]]
gdf_nin = gdf_nin[gdf_nin["nin_int"] != NODATA].reset_index(drop=True)
gdf_nin = gdf_nin.explode(index_parts=False).reset_index(drop=True)
if gdf_nin.crs is None or gdf_nin.crs.to_epsg() != EPSG:
    gdf_nin = gdf_nin.set_crs(CRS)

gdf_nin = mnk.vectorize.merge_slivers(gdf_nin, min_area=2500, max_thinness=0.1, class_col="nin_int")

gdf_nin = gdf_nin.join(mnk.attributes.axes_from_nin_int(gdf_nin["nin_int"])).drop(columns="nin_int")
gdf_nin["datakilde"] = mnk.attributes.build_datakilde(mnk.attributes.SOURCE_SUBSTRAT_NIVA, gdf_nin[TF])

print(f"Polygoner: {len(gdf_nin):,}")
gdf_nin.head()

## Load observation substrate (national)

Substrate for the grid itself comes from the modelled raster in the loop below.
Here only the **observation** polygons (`nisjedata-substrat-klassifisering`) are
loaded; they are applied as a vector overlay after vectorisation to preserve their
geometry.

Observation polygons can overlap, sometimes with different substrate.
`overlay_update` makes the mask non-overlapping first
(`mnk.vectorize.remove_overlaps`): where they overlap, the smaller polygon wins, as
it is the more detailed observation.

In [ ]:
gdf_obs = gpd.read_parquet(obs_substrat_parquet).to_crs(CRS)
gdf_obs = gdf_obs[gdf_obs["BunnType"].isin(BUNN_TO_SUBSTRAT)].reset_index(drop=True)
gdf_obs["_obs_substrat"] = gdf_obs["BunnType"].map(BUNN_TO_SUBSTRAT)
gdf_obs = gdf_obs[["geometry", "_obs_substrat"]]
_ = gdf_obs.sindex  # build spatial index
print(f"Observasjons-polygoner: {len(gdf_obs):,}")

## Tørrfall in the vector step

The tørrfall polygons are small and detailed. Instead of rasterising them to 50 m,
they are applied as a vector overlay: the parts of euphotic, permanently submerged
NiN polygons lying **innenfor** (inside) tørrfall get `nkm_lm-tv` = innenfor and the
kartverket tørrfall source (e.g. `NA-MA05→NA-MA04`, `NA-MC02→NA-MC01`). The afotisk
zone is always sublittoral and is left as is.

The tørrfall polygons do not follow the land mask exactly. The water mask (which
follows the shoreline) is therefore pre-clipped once with tørrfall in
`mnk.sources.prepare_torrfall_filled`: water pieces that become isolated (cut off
from open sea between tørrfall and land) are merged into tørrfall. Open sea is
contiguous and never becomes isolated, so tørrfall never grows seaward.

In [ ]:
# Tørrfall with enclosed water between tørrfall and land merged
gdf_torrfall = gpd.read_parquet(torrfall_parquet).to_crs(CRS)[["geometry"]].reset_index(drop=True)
print(f"Tørrfall-polygoner: {len(gdf_torrfall):,}")

Tørrfall-polygoner: 2,079,804


In [ ]:
gdf_nin_tf = mnk.vectorize.overlay_update(
    gdf_nin,
    gdf_torrfall,
    update=lambda p: p.assign(
        **{TF: mnk.rules.NKM_TOERFALL["innenfor"]},
        datakilde=mnk.attributes.set_kilde(p["datakilde"], mnk.attributes.SOURCE_TOERFALL_KARTVERKET),
    ),
    where=(gdf_nin[LYS] == mnk.rules.NKM_LYS["eufotisk"]) & (gdf_nin[TF] == mnk.rules.NKM_TOERFALL["utenfor"]),
)
print(f"Polygoner etter tørrfall-overlay: {len(gdf_nin_tf):,}")

del gdf_nin, gdf_torrfall
gc.collect()

## Observation substrate in the vector step

The observation polygons are authoritative for substrate. `overlay_update` cuts the
NiN layer along the observation borders, so they are preserved exactly (no pixel
snapping). The pieces inside keep light, salinity and tørrfall from the underlying
polygon; only the substrate axis is set to the observed one, with the NGU source.
`dissolve=True` merges adjacent observation pieces with the same attributes, so the
observations are not split up unnecessarily.

In [ ]:
def set_observed_substrate(pieces):
    return pieces.assign(
        **{SUB: pieces["_obs_substrat"]},
        datakilde=mnk.attributes.set_kilde(pieces["datakilde"], mnk.attributes.SOURCE_SUBSTRAT_NGU),
    )


gdf_nin_obs = mnk.vectorize.overlay_update(gdf_nin_tf, gdf_obs, set_observed_substrate, dissolve=True)
print(f"Polygoner etter observasjons-overlay: {len(gdf_nin_obs):,}")

del gdf_nin_tf, gdf_obs
gc.collect()

## Clip to grunnlinje (1 nautical mile)

Remove polygons outside grunnlinje + 1 nautical mile, before subtracting land.

In [10]:
print(f"Polygoner før grunnlinje-klipp: {len(gdf_nin_obs):,}")
gdf_nin_obs = mnk.vectorize.clip_to_grunnlinje(gdf_nin_obs)
print(f"Polygoner etter grunnlinje-klipp: {len(gdf_nin_obs):,}")

Polygoner før grunnlinje-klipp: 2,846,751


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/osgeo/gdal.py:13117: RuntimeWarning: A geometry of type MULTIPOLYGON is inserted into layer data of geometry type POLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  return _gdal.wrapper_GDALVectorTranslateDestName(*args)


Polygoner etter grunnlinje-klipp: 2,846,271


## Subtract land

In [11]:
land = gpd.read_parquet(land_parquet)
land = land.to_crs(CRS)

gdf_nin_land = mnk.vectorize.subtract_land(gdf_nin_obs, land)
print(f"Polygoner etter land-subtraksjon: {len(gdf_nin_land):,}")

del land, gdf_nin_obs
gc.collect()

Subtracting land from 2,195,292 of 2,846,271 polygons...


Polygoner etter land-subtraksjon: 2,736,485


0

## Checkpoint: clipped layer (parquet)

Saves the clipped layer (axes + `datakilde`) as parquet. Run the **reload** cell below
to rerun the attribute step without rerunning the overlays.


In [ ]:
gdf_nin_land.to_parquet(clipped_checkpoint, compression="snappy")
print(f"Checkpoint lagret: {clipped_checkpoint.name} ({len(gdf_nin_land):,} polygoner)")

## Usikkerhet, NiN type and Naturkart attributes

`mnk.utils.nisje_usikkerhet` gives the per-axis decision-margin estimate (method
`NKM_nisjedata_kvalitet`): a polygon is uncertain when an axis sits near its class
boundary — substrate `|P_soft − P_hard|` (mixture shrinks the margin), light
`|z_eu − depth| / z_eu` (relative), and salinity `|S − 18|`. NGU-observed substrate
forces `lav` on the substrate axis. The classes are kept per axis as
`nkm_usikkerhet_substrat`, `nkm_usikkerhet_lys` and `nkm_usikkerhet_salinitet` (NA
where that axis has no raster data).

`mnk.attributes.classify` then derives `nin_kode_fkid`/`NiN_name` from the axes, and
`mnk.attributes.finalize` derives `nkm_prosess`, `metode_klassifisering` (from
`datakilde`), the combined `nkm_usikkerhetsklasse` (the worst axis) and the canonical
nin_flate_NK column order.

In [4]:
gdf_nin_land = gpd.read_parquet(clipped_checkpoint)
print(f"Lastet checkpoint: {len(gdf_nin_land):,} polygoner")

Lastet checkpoint: 2,736,485 polygoner


In [ ]:
is_obs = (
    gdf_nin_land["datakilde"].str.contains(mnk.attributes.SOURCE_SUBSTRAT_NGU, regex=False).to_numpy()
)
usikkerhet = mnk.utils.nisje_usikkerhet(
    gdf_nin_land,
    kd_path=str(kd_path),
    depth_path=str(depth_path),
    salinity_path=str(salinity_path),
    subprob_path=str(subprob_path),
    is_obs=is_obs,
)
for col in mnk.attributes.USIKKERHET_AXIS_COLS:
    gdf_nin_land[col] = usikkerhet[col].array
gdf_nin_land["metode_usikkerhet"] = mnk.attributes.METODE_USIKKERHET_NISJEDATA

gdf_nin_land = mnk.attributes.finalize(mnk.attributes.classify(gdf_nin_land))
for col in ["nkm_usikkerhetsklasse", *mnk.attributes.USIKKERHET_AXIS_COLS]:
    print(gdf_nin_land[col].value_counts(dropna=False), "\n")
gdf_nin_land.head()

## Save result

In [ ]:
fname = mnk.utils.to_filename("nkm-utbredelsesmodell", "norge", "2026", CRS.split(":")[1])
out_dir = REPO

gdf_nin_land.to_parquet(out_dir / f"{fname}.geo.parquet", compression="snappy")
print(f"Lagret {len(gdf_nin_land):,} polygoner -> {fname}.geo.parquet")

In [16]:
# Upload to PostGIS (optional)
mnk.utils.to_postgis(gdf_nin_land, fname)

NIVAGIS_CONNECTION_STR not set. Skipping PostGIS upload.
